In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [15]:
df = pd.read_excel('../data/Consolidated Sales.xlsx', sheet_name='Main')

KeyboardInterrupt: 

In [5]:
DROP_COLS = ['Billing Doc.', 'Item', 'Stor. Location', 'Sales district','Dealer Name','Customer', 'Material.1','Segment.1',
       'Basic Value', 'Deal Cat1', 'USD Value', 'Deal Cat2', 'Tax Amount',
       'Price Group', 'BDC Discount', 'EDC Discount', 'Sale Supp Discount',
       'Trade Discount', 'ZDSD Discount', 'Fest Discount', 'ZDIN Spl Discount',
       'SLC/DC/Mat','Mat/DC/SO', 'Mat/DC', 'ZDI8-SOrg/Cust',
       'ZDI9-SOrg/RegMarket', 'RR Discount', 'Sol Plaza Discount', 'Turn Over',
       'Sales Commission', 'Cash Discount', 'Add Cash Discount', 'Sales Order',
       'Order Date','SSS Discount', 'GR Total', 'Turn Over-CD', 'Ord Reason', 'Sord Created by', 'Dealer Code', 'PayTerm', 'PersonRes',
       'Order Type', 'YourRef', 'Distribution Channel Billing',
       'Sold-to party', 'Discount','Cust Grp','Qt No.', 'Qt Type', 'Qt Date', 'MAT_GP7', 'SH CITY', 'SH POCODE','SH STATE','Sales Person Name', 'Deal/Cust','Application Name', 'Billing', 'Inter/Intra', 'Period']
dropped_df = df.drop(columns=DROP_COLS)

In [136]:
new_df = df.copy()
new_df = new_df[new_df['Approval Number'] != 'SALES RETURN']
new_df = new_df[(new_df['Billing Quantity ODU'] > 0) | (new_df['Billing Quantity IDU'] > 0)] 
new_df['Billing Doc.'] = new_df['Billing Doc.'].astype(str)
new_df['Unit Type'] = 'Unknown'
new_df['Action'] = 'Unknown'
new_df.shape

(318001, 87)

In [142]:
# First, set an index for faster lookups
left = new_df.set_index(['Billing Doc.', 'Deal/Cust', 'Billing Date', 'Billing Quantity IDU'])
right = new_df.set_index(['Billing Doc.', 'Deal/Cust', 'Billing Date', 'Billing Quantity ODU'])

# Align the indices
matches = left.join(right[['Tonnage']], how='left', rsuffix='_match')

# Update columns
matches['Tonnage'] = matches['Tonnage_match']
matches['Unit Type'] = 'IDU'
matches['Action'] = 'Keep'

# Mark the matched rows in the original dataframe as Delete
delete_index = right.index.intersection(left.index)
new_df.loc[new_df.set_index(['Billing Doc.', 'Deal/Cust', 'Billing Date', 'Billing Quantity ODU']).index.isin(delete_index), 'Action'] = 'Delete'


In [145]:
new_df['Action'].value_counts()


Action
Delete     314913
Unknown      3088
Name: count, dtype: int64

In [305]:
refactored_df = dropped_df[['Year', 'Month',
       'Week', 'Billing Date', 'Material', 'Division', 'SALES OFFICE CODE', 'Plant', 'EXT MAT GROUP', 'SBU', 'MatGroup3',
       'Star Rating', 'Segment', 'Star Rating.1',   'Billing Quantity ODU', 'Billing Quantity IDU',
       'Tonnage', 'CustCity', 'Location',
       'Location State', 'Plant Zone',  
       'State Zone', 'Sales Group', 'Approval Number']]
refactored_df = refactored_df[refactored_df['Approval Number'] != 'SALES RETURN']
refactored_df = refactored_df.drop(columns=['Approval Number', 'Sales Group'])
refactored_df = refactored_df[(refactored_df['Billing Quantity ODU'] >= 0) & (refactored_df['Billing Quantity IDU'] >= 0) & (refactored_df['Billing Date'] <= '2024-03-30')]
refactored_df['Qty'] = refactored_df['Billing Quantity ODU'] + refactored_df['Billing Quantity IDU']
refactored_df['Unit Type'] = refactored_df['Billing Quantity IDU'].apply(lambda x: 'IDU' if x > 0 else 'ODU')
refactored_df = refactored_df.drop(columns=['Billing Quantity ODU', 'Billing Quantity IDU'])
refactored_df = refactored_df[refactored_df['State Zone'] == 'South']
refactored_df = refactored_df[refactored_df['Plant Zone'] == 'South']
refactored_df = refactored_df.drop(columns=['Plant Zone', 
                                            'State Zone', 
                                            'CustCity',
                                            'Material', 
                                            'Plant',
                                            'Division', 
                                            'Star Rating', 
                                            'MatGroup3', 
                                            'SBU', 
                                            'EXT MAT GROUP',
                                            'Location State',
                                            'Location'])
refactored_df['Year'] = refactored_df['Billing Date'].dt.year
refactored_df['Month'] = refactored_df['Billing Date'].dt.month
refactored_df['Week'] = refactored_df['Billing Date'].dt.isocalendar().week
refactored_df.sort_values(by=['Billing Date'], inplace=True)
refactored_df.reset_index(drop=True, inplace=True)
refactored_df.rename(columns={'Billing Date': 'Date', 'SALES OFFICE CODE': 'Branch', 'Star Rating.1': 'Rating'}, inplace=True)
refactored_df['Rating'] = refactored_df['Rating'].replace('3 star', '3 Star')
refactored_df['Tonnage'] = refactored_df['Tonnage'].replace(0.75, 0.8)
refactored_df['Tonnage'] = refactored_df['Tonnage'].replace(2.20, 2.0)
refactored_df['Tonnage'] = refactored_df['Tonnage'].replace(2.35, 2.0)
refactored_df['Tonnage'] = refactored_df['Tonnage'].replace(2.50, 2.0)


In [186]:
refactored_df.head()


,Year,Month,Week,Date,Branch,Segment,Rating,Tonnage,Qty,Unit Type
0,2019,4,14,2019-04-03,MAA,Non Inv,3 Star,1.5,10.0,ODU
1,2019,4,14,2019-04-03,BLR,Inverter,5 Star,0.0,11.0,IDU
2,2019,4,14,2019-04-03,MAA,Non Inv,3 Star,0.0,10.0,IDU
3,2019,4,14,2019-04-03,MAA,Non Inv,3 Star,1.0,10.0,ODU
4,2019,4,14,2019-04-03,MAA,Non Inv,3 Star,0.0,10.0,IDU


In [188]:
print(refactored_df['Branch'].value_counts())
print("="*100)
print(refactored_df['Segment'].value_counts())
print("="*100)
print(refactored_df['Rating'].value_counts())
print("="*100)
print(refactored_df['Tonnage'].value_counts())
print("="*100)
print(refactored_df['Unit Type'].value_counts())

Branch
MAA     114122
COK      51161
SBD1     44745
SBD      42077
BLR      35223
Name: count, dtype: int64
Segment
Inverter    209089
Non Inv      78239
Name: count, dtype: int64
Rating
3 Star    194136
5 Star     77370
2 Star      8470
1 Star      4608
4 Star      2744
Name: count, dtype: int64
Tonnage
0.0    143553
1.5     72193
1.0     38189
1.8     19911
2.0      8092
0.8      5390
Name: count, dtype: int64
Unit Type
ODU    144131
IDU    143197
Name: count, dtype: int64


In [193]:
print(refactored_df[refactored_df['Tonnage'] == 0]['Unit Type'].value_counts())


Unit Type
IDU    143197
ODU       356
Name: count, dtype: int64
